# Vesper Capital — Live Signal Monitor

Daily signal generation with GitHub-verified timestamps.

**Why this matters:** the walk-forward validation is rigorous historical evidence (p=0.028) but it is still backward-looking. A live track record with timestamped commits is a fundamentally different kind of evidence — each commit is cryptographic proof that a prediction was made before the 24-hour price move resolved.

**Architecture:**
1. Train model once on full validation window (Jan 2023 - Dec 2024) → pickle
2. Daily: fetch latest balances, build features, predict
3. Write signal to markdown + append to CSV
4. Git commit to vesper_capital repo
5. After resolution (24h later): result verifiable on-chain

**Schedule:** runs daily at 8 AM UTC via cron. Each run takes ~3-5 minutes.

**Cost:** roughly 200 wallets × 1 hour × $0.000003/balance ≈ negligible per day.

**What this gives you:**
- 30 days → 30 verified out-of-sample signals
- 90 days → statistically meaningful forward record
- 180 days → credible track record for commercialization

## 1. Configuration

In [ ]:
from claude_refactor_core_utils import *
import pandas as pd
import numpy as np
import os
import json
import pickle
import shutil
import subprocess
from datetime import date, datetime, timedelta
from pathlib import Path
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────
# Repo root — all committed outputs live under here
BASE_DIR        = Path('/home/colin/projects/blockchain_alpha/vesper_capital')
LIVE_DIR        = BASE_DIR / 'data/live_signals'
LIVE_DIR.mkdir(parents=True, exist_ok=True)

# Balance store lives one level up (not in the repo — gitignored large file)
HOURLY_BAL_CSV  = BASE_DIR.parent / 'data/balance_store_hourly_top200.csv'

MODEL_PKL        = LIVE_DIR / 'live_model.pkl'
FEATURE_COLS_PKL = LIVE_DIR / 'feature_cols.pkl'
MODEL_ARCHIVE    = LIVE_DIR / 'model_archive'          # local only, gitignored
MODEL_ARCHIVE.mkdir(parents=True, exist_ok=True)

DAILY_SIGNAL_CSV = LIVE_DIR / 'daily_signals.csv'
TODAY_SIGNAL_MD  = LIVE_DIR / f'signal_{date.today().strftime("%Y-%m-%d")}.md'

# ── Config ───────────────────────────────────────────────────
ALCHEMY_API_KEY = "8J0Ou3_XsuWVbOiZ0xpEvioGx7k0wab9"

# Use baseline 200-wallet pool — validated p=0.028
# Train on full validation window (Jan 2023 - Dec 2024)
TRAIN_START     = date(2023, 1, 1)
TRAIN_END       = date(2024, 12, 31)

# Signal generation params (from validated walk-forward)
MIN_THRESHOLD   = 0.50
MIN_DELTA_ETH   = 0.005
ROLLING_WINDOWS = [6, 24, 72, 168]

# Model params (matches walk-forward)
MODEL_PARAMS = dict(
    n_estimators=200, max_depth=3, learning_rate=0.02,
    subsample=0.6, colsample_bytree=0.3, min_child_weight=50,
    reg_alpha=1.0, reg_lambda=5.0, random_state=42,
    n_jobs=-1, verbosity=0
)

print("=" * 60)
print("VESPER CAPITAL — LIVE SIGNAL MONITOR")
print("=" * 60)
print()
print(f"Today:              {date.today()}")
print(f"Training window:    {TRAIN_START} to {TRAIN_END}")
print(f"Wallet pool:        validated 200-wallet baseline")
print(f"Min threshold:      {MIN_THRESHOLD}")
print(f"Repo root:          {BASE_DIR}")
print(f"Balance store:      {HOURLY_BAL_CSV}")
print(f"Output dir:         {LIVE_DIR}")


## 2. Train production model

Trained once on full validation window. Cached as pickle for daily reuse. Auto-retrains every 30 days.

On retrain: the previous `live_model.pkl` is archived locally as `model_archive/live_model_vN_YYYYMMDD.pkl` before being overwritten. Archived models are gitignored — only the current production model is pushed to the remote.

In [ ]:
# ============================================================
# Step 1: Train production model on full validation window
# Reuses already-trained model if pickle exists and < 30 days old
# On retrain: archives previous model locally before overwriting
# ============================================================

def _next_model_version(archive_dir: Path) -> int:
    """Return next integer model version based on archived filenames."""
    existing = sorted(archive_dir.glob('live_model_v*.pkl'))
    if not existing:
        return 2   # first archive = v2 (v1 was the original)
    last = existing[-1].stem  # e.g. 'live_model_v3_20260401'
    try:
        return int(last.split('_v')[1].split('_')[0]) + 1
    except (IndexError, ValueError):
        return len(existing) + 2

retrain = True
if MODEL_PKL.exists() and FEATURE_COLS_PKL.exists():
    age_days = (datetime.now() - datetime.fromtimestamp(MODEL_PKL.stat().st_mtime)).days
    if age_days < 30:
        print(f"Loading existing model (age: {age_days} days)")
        with open(MODEL_PKL, "rb") as f:
            model = pickle.load(f)
        with open(FEATURE_COLS_PKL, "rb") as f:
            FEATURE_COLS = pickle.load(f)
        retrain = False
    else:
        print(f"Model is {age_days} days old — archiving and retraining")
        # Archive current model before overwriting
        v = _next_model_version(MODEL_ARCHIVE)
        stamp = datetime.now().strftime('%Y%m%d')
        archived_model = MODEL_ARCHIVE / f'live_model_v{v}_{stamp}.pkl'
        archived_cols  = MODEL_ARCHIVE / f'feature_cols_v{v}_{stamp}.pkl'
        shutil.copy2(MODEL_PKL, archived_model)
        shutil.copy2(FEATURE_COLS_PKL, archived_cols)
        print(f"  Archived as: {archived_model.name}")

if retrain:
    print("Training production model...")
    print(f"  Loading {HOURLY_BAL_CSV.name}...")

    raw = pd.read_csv(HOURLY_BAL_CSV)
    raw['date']        = pd.to_datetime(raw['date'], format='mixed').dt.floor('h')
    raw['address']     = raw['address'].str.lower().str.strip()
    raw['balance_wei'] = pd.to_numeric(raw['balance_wei'], errors='coerce').fillna(0).astype('int64')

    hourly = (raw.sort_values(['address','date','block_number'])
              .groupby(['address','date'], as_index=False).tail(1)
              .reset_index(drop=True))
    hourly['balance_eth'] = hourly['balance_wei'] / 1e18
    hourly = hourly.sort_values(['address','date'])
    hourly['delta_eth']  = hourly.groupby('address')['balance_eth'].diff()
    hourly['action_raw'] = hourly['delta_eth'].apply(
        lambda d: 1 if d > MIN_DELTA_ETH else (-1 if d < -MIN_DELTA_ETH else 0)
    )

    wallet_list = sorted(hourly['address'].unique().tolist())
    print(f"  Wallet pool: {len(wallet_list)} wallets")

    # Price data
    print(f"  Loading price data...")
    df_price = load_eth_usd_cryptocompare(
        start=str(TRAIN_START), end=str(TRAIN_END), interval='1h'
    )
    df_price['date'] = pd.to_datetime(df_price['date']).dt.floor('h')
    df_price = df_price.sort_values('date').reset_index(drop=True)
    df_price['fwd_return'] = (
        df_price['close'].shift(-24) - df_price['close']
    ) / df_price['close'] * 100
    df_price['ret_1h']      = df_price['close'].pct_change(1)  * 100
    df_price['ret_4h']      = df_price['close'].pct_change(4)  * 100
    df_price['ret_24h']     = df_price['close'].pct_change(24) * 100
    df_price['ret_72h']     = df_price['close'].pct_change(72) * 100
    df_price['vol_24h']     = df_price['ret_1h'].rolling(24).std()
    df_price['vol_72h']     = df_price['ret_1h'].rolling(72).std()
    df_price['hour_of_day'] = df_price['date'].dt.hour
    df_price['day_of_week'] = df_price['date'].dt.dayofweek

    # Build features
    print(f"  Building feature matrix...")
    action_wide = hourly.pivot_table(
        index='date', columns='address', values='action_raw', aggfunc='first'
    ).fillna(0)
    delta_wide = hourly.pivot_table(
        index='date', columns='address', values='delta_eth', aggfunc='first'
    ).fillna(0)

    wallet_features = pd.DataFrame(index=action_wide.index)
    for addr in wallet_list:
        if addr not in action_wide.columns:
            continue
        short = addr[:10]
        act   = action_wide[addr]
        dlt   = delta_wide[addr]
        wallet_features[f'{short}_action']     = act
        wallet_features[f'{short}_wtd_action'] = dlt * act.abs()
        for w in ROLLING_WINDOWS:
            wallet_features[f'{short}_buy_pressure_{w}h'] = (
                (act > 0).astype(float).rolling(w, min_periods=1).mean()
            )
        wallet_features[f'{short}_max_buy_24h'] = dlt.clip(lower=0).rolling(24, min_periods=1).max()

    buy_matrix  = (action_wide > 0).astype(float)
    sell_matrix = (action_wide < 0).astype(float)
    wallet_features['pool_buy_fraction']  = buy_matrix.mean(axis=1)
    wallet_features['pool_sell_fraction'] = sell_matrix.mean(axis=1)
    wallet_features['pool_net_fraction']  = wallet_features['pool_buy_fraction'] - wallet_features['pool_sell_fraction']
    buy_pressure  = delta_wide.clip(lower=0).sum(axis=1)
    sell_pressure = delta_wide.clip(upper=0).abs().sum(axis=1)
    total_vol     = buy_pressure + sell_pressure + 1e-9
    wallet_features['pool_net_pressure_eth'] = (buy_pressure - sell_pressure) / total_vol
    for w in [6, 24, 72]:
        wallet_features[f'pool_buy_fraction_{w}h'] = (
            wallet_features['pool_buy_fraction'].rolling(w, min_periods=1).mean()
        )
        wallet_features[f'pool_net_pressure_{w}h'] = (
            wallet_features['pool_net_pressure_eth'].rolling(w, min_periods=1).mean()
        )
    n_active = (action_wide != 0).sum(axis=1).clip(lower=1)
    n_agree  = buy_matrix.sum(axis=1).combine(sell_matrix.sum(axis=1), max)
    wallet_features['pool_conviction'] = n_agree / n_active

    feature_df = wallet_features.reset_index().rename(columns={'date':'timestamp'})
    price_cols = ['date','close','fwd_return',
                  'ret_1h','ret_4h','ret_24h','ret_72h',
                  'vol_24h','vol_72h','hour_of_day','day_of_week']
    full_df = feature_df.merge(
        df_price[price_cols].rename(columns={'date':'timestamp'}),
        on='timestamp', how='inner'
    ).dropna(subset=['fwd_return']).dropna().reset_index(drop=True)

    EXCLUDE_COLS = ['timestamp','close','fwd_return']
    FEATURE_COLS = [c for c in full_df.columns if c not in EXCLUDE_COLS]
    print(f"  Features: {len(FEATURE_COLS)}, Rows: {len(full_df):,}")

    # Train on full window
    X = full_df[FEATURE_COLS].values
    y = full_df['fwd_return'].values

    model = xgb.XGBRegressor(**MODEL_PARAMS)
    model.fit(X, y)

    # Save current production model (overwrites previous)
    with open(MODEL_PKL, "wb") as f:
        pickle.dump(model, f)
    with open(FEATURE_COLS_PKL, "wb") as f:
        pickle.dump(FEATURE_COLS, f)

    print(f"  Model trained and saved to {MODEL_PKL.name}")
    print(f"  Trained on {len(X):,} rows × {len(FEATURE_COLS)} features")
    print(f"  Model archive: {len(list(MODEL_ARCHIVE.glob('live_model_v*.pkl')))} previous version(s) stored locally")


## 3. Fetch latest balance data

Updates the hourly balance store with any new hours since last run. Restart-safe — only fetches missing data.

In [ ]:
# ============================================================
# Step 2: Generate signal for current hour
# Fetches latest hour of balance data, builds features,
# runs through model, writes signal output
# ============================================================

# Fetch most recent balance snapshot for all wallets
print("Fetching current wallet state...")
current_hour = pd.Timestamp.utcnow().floor('h').tz_localize(None)
lookback_start = current_hour - pd.Timedelta(hours=ROLLING_WINDOWS[-1] + 24)

# Need recent history for rolling features
# Load existing hourly store and append recent hours
raw = pd.read_csv(HOURLY_BAL_CSV)
raw['date']    = pd.to_datetime(raw['date'], format='mixed').dt.floor('h')
raw['address'] = raw['address'].str.lower().str.strip()
raw['balance_wei'] = pd.to_numeric(raw['balance_wei'], errors='coerce').fillna(0).astype('int64')

wallet_list = sorted(raw['address'].unique().tolist())
last_known_date = raw['date'].max()
print(f"  Existing data through: {last_known_date}")
print(f"  Current hour:          {current_hour}")
print(f"  Wallets to update:     {len(wallet_list)}")

# Build hourly schedule from last_known_date + 1h through current_hour
hours_to_fetch = pd.date_range(
    start=last_known_date + pd.Timedelta(hours=1),
    end=current_hour, freq='h'
)
print(f"  Hours to fetch:        {len(hours_to_fetch)}")

if len(hours_to_fetch) > 0:
    # Load block map and extend if needed
    bm_df = pd.read_csv(BASE_DIR.parent / 'data/hourly_block_map.csv')
    bm_df['timestamp']    = pd.to_datetime(bm_df['timestamp']).dt.floor('h')
    bm_df['block_number'] = bm_df['block_number'].astype(int)

    # Get current block
    import requests
    BASE_URL = f"https://eth-mainnet.g.alchemy.com/v2/{ALCHEMY_API_KEY}"
    cb_resp  = requests.post(BASE_URL, json={
        "jsonrpc":"2.0","method":"eth_blockNumber","params":[],"id":1
    }, timeout=10)
    current_block = int(cb_resp.json()["result"], 16)

    last_bm_time  = bm_df['timestamp'].max()
    last_bm_block = int(bm_df[bm_df['timestamp']==last_bm_time]['block_number'].iloc[0])

    if current_hour > last_bm_time:
        hours_gap = int((current_hour - last_bm_time).total_seconds() / 3600)
        bph = (current_block - last_bm_block) / hours_gap
        new_rows = []
        for h in range(1, hours_gap + 1):
            new_rows.append({
                'timestamp': last_bm_time + pd.Timedelta(hours=h),
                'block_number': int(last_bm_block + bph * h)
            })
        bm_df = pd.concat([bm_df, pd.DataFrame(new_rows)], ignore_index=True)
        bm_df.to_csv(BASE_DIR.parent / 'data/hourly_block_map.csv', index=False)
        print(f"  Block map extended by {hours_gap}h to {bm_df['timestamp'].max()}")

    # Build schedule and fetch
    schedule = pd.DataFrame([
        {'address': addr, 'date': h}
        for addr in wallet_list for h in hours_to_fetch
    ])
    window_bm = bm_df[
        (bm_df['timestamp'] >= hours_to_fetch.min()) &
        (bm_df['timestamp'] <= hours_to_fetch.max())
    ].reset_index(drop=True)

    print(f"  Fetching {len(schedule):,} balance snapshots...")
    new_bal = fetch_balances_for_schedule(
        schedule=schedule, block_df=window_bm,
        alchemy_api_key=ALCHEMY_API_KEY,
        batch_size=100, max_concurrency=20, rps=25.0,
        progress_every=99999, block_date_col='timestamp',
        label=f"live-{current_hour.strftime('%Y%m%d-%H')}",
    )
    new_bal['date']    = pd.to_datetime(new_bal['date'], format='mixed').dt.floor('h')
    new_bal['address'] = new_bal['address'].str.lower().str.strip()
    new_bal['balance_wei'] = pd.to_numeric(new_bal['balance_wei'], errors='coerce').fillna(0).astype('int64')

    # Append to existing store
    combined = pd.concat([raw, new_bal], ignore_index=True)
    combined = (combined.sort_values(['address','date','block_number'])
                .groupby(['address','date'], as_index=False).tail(1)
                .reset_index(drop=True))
    combined.to_csv(HOURLY_BAL_CSV, index=False)
    print(f"  Balance store updated: {len(combined):,} rows")
    raw = combined
else:
    print("  Balance store already current")

print(f"\nGenerating signal for {current_hour}...")


## 4. Build features and predict

Constructs the feature row for the current hour and runs it through the model.

In [ ]:
# Build feature matrix for recent window (last 7 days)
recent_start = current_hour - pd.Timedelta(hours=ROLLING_WINDOWS[-1])
recent = raw[raw['date'] >= recent_start].copy()
recent['balance_eth'] = recent['balance_wei'] / 1e18
recent = recent.sort_values(['address','date'])
recent['delta_eth']  = recent.groupby('address')['balance_eth'].diff()
recent['action_raw'] = recent['delta_eth'].apply(
    lambda d: 1 if d > MIN_DELTA_ETH else (-1 if d < -MIN_DELTA_ETH else 0)
)
wallet_list = sorted(recent['address'].unique().tolist())

action_wide = recent.pivot_table(
    index='date', columns='address', values='action_raw', aggfunc='first'
).fillna(0)
delta_wide = recent.pivot_table(
    index='date', columns='address', values='delta_eth', aggfunc='first'
).fillna(0)

wf = pd.DataFrame(index=action_wide.index)
for addr in wallet_list:
    if addr not in action_wide.columns:
        continue
    short = addr[:10]
    act   = action_wide[addr]
    dlt   = delta_wide[addr]
    wf[f'{short}_action']     = act
    wf[f'{short}_wtd_action'] = dlt * act.abs()
    for w in ROLLING_WINDOWS:
        wf[f'{short}_buy_pressure_{w}h'] = (
            (act > 0).astype(float).rolling(w, min_periods=1).mean()
        )
    wf[f'{short}_max_buy_24h'] = dlt.clip(lower=0).rolling(24, min_periods=1).max()

bm_now = (action_wide > 0).astype(float)
sm_now = (action_wide < 0).astype(float)
wf['pool_buy_fraction']  = bm_now.mean(axis=1)
wf['pool_sell_fraction'] = sm_now.mean(axis=1)
wf['pool_net_fraction']  = wf['pool_buy_fraction'] - wf['pool_sell_fraction']
bp = delta_wide.clip(lower=0).sum(axis=1)
sp = delta_wide.clip(upper=0).abs().sum(axis=1)
tv = bp + sp + 1e-9
wf['pool_net_pressure_eth'] = (bp - sp) / tv
for w in [6, 24, 72]:
    wf[f'pool_buy_fraction_{w}h'] = wf['pool_buy_fraction'].rolling(w, min_periods=1).mean()
    wf[f'pool_net_pressure_{w}h'] = wf['pool_net_pressure_eth'].rolling(w, min_periods=1).mean()
n_act = (action_wide != 0).sum(axis=1).clip(lower=1)
n_agr = bm_now.sum(axis=1).combine(sm_now.sum(axis=1), max)
wf['pool_conviction'] = n_agr / n_act

feature_df = wf.reset_index().rename(columns={'date':'timestamp'})

# Price features for current hour
df_price_recent = load_eth_usd_cryptocompare(
    start=str(recent_start.date()),
    end=str(current_hour.date() + pd.Timedelta(days=1).to_pytimedelta()),
    interval='1h'
)
df_price_recent['date']        = pd.to_datetime(df_price_recent['date']).dt.floor('h')
df_price_recent                = df_price_recent.sort_values('date').reset_index(drop=True)
df_price_recent['ret_1h']      = df_price_recent['close'].pct_change(1)  * 100
df_price_recent['ret_4h']      = df_price_recent['close'].pct_change(4)  * 100
df_price_recent['ret_24h']     = df_price_recent['close'].pct_change(24) * 100
df_price_recent['ret_72h']     = df_price_recent['close'].pct_change(72) * 100
df_price_recent['vol_24h']     = df_price_recent['ret_1h'].rolling(24).std()
df_price_recent['vol_72h']     = df_price_recent['ret_1h'].rolling(72).std()
df_price_recent['hour_of_day'] = df_price_recent['date'].dt.hour
df_price_recent['day_of_week'] = df_price_recent['date'].dt.dayofweek

price_cols = ['date','close','ret_1h','ret_4h','ret_24h','ret_72h',
              'vol_24h','vol_72h','hour_of_day','day_of_week']
live_df = feature_df.merge(
    df_price_recent[price_cols].rename(columns={'date':'timestamp'}),
    on='timestamp', how='inner'
).dropna()

# Filter to current hour only for prediction
current_row = live_df[live_df['timestamp'] == current_hour]
if len(current_row) == 0:
    # Use most recent available row
    current_row = live_df.tail(1)
    print(f"  Note: using most recent available hour {current_row['timestamp'].iloc[0]}")

# Align features with training feature set
X_live = pd.DataFrame(columns=FEATURE_COLS)
for col in FEATURE_COLS:
    X_live[col] = current_row[col].values if col in current_row.columns else [0.0]
X_live = X_live.fillna(0.0)

# Predict
y_pred = float(model.predict(X_live.values)[0])
ts_pred = current_row['timestamp'].iloc[0]
close_pred = float(current_row['close'].iloc[0])
pool_buy_frac = float(current_row['pool_buy_fraction'].iloc[0])
pool_net_pressure = float(current_row['pool_net_pressure_eth'].iloc[0])

# Signal decision
fires = y_pred > MIN_THRESHOLD
direction = "LONG" if fires else "FLAT"

print(f"\n{'='*60}")
print(f"SIGNAL FOR {ts_pred}")
print(f"{'='*60}")
print(f"  Model prediction:      {y_pred:+.4f}%")
print(f"  Threshold:             {MIN_THRESHOLD:.2f}%")
print(f"  Signal fires:          {fires}")
print(f"  Direction:             {direction}")
print()
print(f"  ETH close:             ${close_pred:,.2f}")
print(f"  Pool buy fraction:     {pool_buy_frac:.4f}")
print(f"  Pool net pressure:     {pool_net_pressure:+.4f}")


## 5. Persist signal

Appends to running CSV and writes today's signal as markdown for the repo.

In [ ]:
# ============================================================
# Step 3: Persist signal to daily log + GitHub commit
# ============================================================

signal_record = {
    'timestamp':           ts_pred.isoformat(),
    'generated_at':        datetime.utcnow().isoformat(),
    'eth_close':           close_pred,
    'predicted_24h_return':y_pred,
    'threshold':           MIN_THRESHOLD,
    'signal_fires':        bool(fires),
    'direction':           direction,
    'pool_buy_fraction':   pool_buy_frac,
    'pool_net_pressure':   pool_net_pressure,
    'wallet_count':        len(wallet_list),
    'model_version':       'v1_200wallet_baseline',
}

# Append to daily signals CSV
new_row_df = pd.DataFrame([signal_record])
if DAILY_SIGNAL_CSV.exists():
    existing = pd.read_csv(DAILY_SIGNAL_CSV)
    combined_signals = pd.concat([existing, new_row_df], ignore_index=True)
    combined_signals = combined_signals.drop_duplicates(subset=['timestamp'], keep='last')
    combined_signals.to_csv(DAILY_SIGNAL_CSV, index=False)
else:
    new_row_df.to_csv(DAILY_SIGNAL_CSV, index=False)

print(f"Signal appended to {DAILY_SIGNAL_CSV.name}")
print(f"  Total signals logged: {len(pd.read_csv(DAILY_SIGNAL_CSV))}")

# Write today's signal as readable markdown
md = f"""# Vesper Capital Live Signal

**Generated:** {datetime.utcnow().isoformat()}
**Signal timestamp:** {ts_pred}

## Prediction

| Field | Value |
|-------|-------|
| Predicted 24h return | **{y_pred:+.4f}%** |
| Threshold | {MIN_THRESHOLD:.2f}% |
| Signal fires | {'**YES**' if fires else 'NO'} |
| Direction | **{direction}** |

## Market Context

| Field | Value |
|-------|-------|
| ETH close | ${close_pred:,.2f} |
| Pool buy fraction | {pool_buy_frac:.4f} |
| Pool net pressure | {pool_net_pressure:+.4f} |
| Wallet count | {len(wallet_list)} |

## Model

- Version: v1 (200-wallet validated baseline)
- Trained: {TRAIN_START} to {TRAIN_END}
- Walk-forward p-value: 0.028 (non-November folds)
- Validated mean edge: +1.005%/hr

## Verification

This signal was generated and committed before the 24-hour return window resolved.
Cryptographic proof of timing: see git commit hash and timestamp.

To verify: check ETH price at {ts_pred} (generation time) vs. {ts_pred + pd.Timedelta(hours=24)} (resolution time).
"""

with open(TODAY_SIGNAL_MD, "w") as f:
    f.write(md)
print(f"Markdown signal written to {TODAY_SIGNAL_MD.name}")


## 6. Commit to GitHub

The commit hash + timestamp is the cryptographic proof that this signal was generated BEFORE the 24h price move resolved.

On retrain days, `live_model.pkl` and `feature_cols.pkl` are also staged so the current production model is always in the remote.

In [ ]:
# ============================================================
# Step 4: Commit to GitHub for verifiable timestamp
# ============================================================

def git_commit_and_push(repo_dir, files, message):
    """Stage, commit, and push files to GitHub."""
    cmds = [
        ["git", "-C", str(repo_dir), "add"] + [str(f) for f in files],
        ["git", "-C", str(repo_dir), "commit", "-m", message],
        ["git", "-C", str(repo_dir), "push"],
    ]
    for cmd in cmds:
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0 and "nothing to commit" not in result.stdout.lower():
            print(f"  Git command failed: {' '.join(cmd)}")
            print(f"  stderr: {result.stderr}")
            return False
    return True

print("Committing signal to GitHub...")

# Always commit signal files
files_to_commit = [DAILY_SIGNAL_CSV, TODAY_SIGNAL_MD]

# Also commit updated model on retrain days
if retrain:
    files_to_commit += [MODEL_PKL, FEATURE_COLS_PKL]
    print("  Retrain day — including updated model in commit")

commit_msg = (
    f"Live signal {ts_pred.strftime('%Y-%m-%d %H:%M')} "
    f"| {direction} | pred={y_pred:+.4f}% | ETH=${close_pred:,.0f}"
)
if retrain:
    commit_msg += " | model retrained"

success = git_commit_and_push(BASE_DIR, files_to_commit, commit_msg)
if success:
    # Get commit hash for verification
    hash_result = subprocess.run(
        ["git", "-C", str(BASE_DIR), "rev-parse", "HEAD"],
        capture_output=True, text=True
    )
    commit_hash = hash_result.stdout.strip()
    print(f"  Committed: {commit_hash[:12]}")
    print(f"  Message:   {commit_msg}")
else:
    print("  Commit failed — check git config and remote")
    print(f"  Run manually: git -C {BASE_DIR} status")


## 7. Cron setup

Instructions for deploying as a daily cron job.

In [ ]:
# ============================================================
# Step 5: Cron setup instructions
# ============================================================

print("=" * 65)
print("DEPLOYMENT — CRON JOB SETUP")
print("=" * 65)
print()
print("Repo root: /home/colin/projects/blockchain_alpha/vesper_capital")
print()
print("1. Wrapper script (already created by deploy.sh):")
print("   /home/colin/projects/blockchain_alpha/vesper_capital/run_live_signal.sh")
print()
print("2. Crontab entry (runs daily at 8 AM UTC):")
print("   0 8 * * * /home/colin/projects/blockchain_alpha/vesper_capital/run_live_signal.sh")
print()
print("3. Verify GitHub remote:")
print("   git -C /home/colin/projects/blockchain_alpha/vesper_capital remote -v")
print()
print("4. Monitor logs:")
print("   tail -f /home/colin/projects/blockchain_alpha/vesper_capital/data/live_signals/cron.log")
print()
print("=" * 65)
print("WHAT THIS GIVES YOU")
print("=" * 65)
print()
print("After 30 days:  30 signals committed before their 24h resolution")
print("After 90 days:  Statistically meaningful forward record")
print("After 180 days: Credible track record for licensees/investors")
print()
print("Each commit is a cryptographically timestamped signal that")
print("CANNOT be backdated. The GitHub commit hash + timestamp is")
print("verifiable proof that the prediction was made BEFORE the")
print("24h price move resolved.")
